[Reference](https://medium.com/@umairali.khan/building-a-generic-knowledge-extraction-ai-framework-for-organization-specific-use-cases-cbb52ce93e48)

In [1]:
requirements = """
Extract project information from research grant proposals:
- Project title (string, required)
- Total budget in EUR (decimal, required)
- Start date (date, format: iso-date)
- Project status (enum: active, completed, pending)
- Principal investigator name (string)
"""

In [5]:
from __future__ import annotations

import json
from typing import List, Dict, Any, Optional
from pydantic import BaseModel

In [6]:
class FieldSpec(BaseModel):
    field_name: str                     # snake_case via validator
    field_type: AllowedTypes            # "str", "int", "decimal", "list[str]", etc.
    description: str
    required: bool
    enum: Optional[list[str]] = None
    pattern: Optional[str] = None
    format: Optional[Literal["iso-date", "currency-eur"]] = None

In [7]:
class ExtractionRequirements(BaseModel):
    use_case_name: str
    fields: list[FieldSpec]

In [8]:
class StructureAnalysis(BaseModel):
    structure_type: Literal["flat", "nested_list"]
    parent_container_name: str  # e.g., "line_items"
    parent_description: str
    item_description: str
    reasoning: str

In [9]:
prompt = """
Analyze the following extraction requirements and determine the output structure.

Use NESTED_LIST when:
- The DOCUMENT contains multiple items/records/rows to extract
- Instructions mention 'multiple items IN THE DOCUMENT', 'list of items', 'table of records'
- 'one line per item', 'one row per record', 'repeat for each entry'
- Document is structured as a table, list, or collection of similar items
- Example: Extract all products from an invoice (multiple products in one invoice)

Use FLAT when:
- ONE record per document (even if processing multiple documents)
- 'for each document', 'from each document', 'per document'
- Document describes a SINGLE entity (e.g., one project, one invoice, one person)
- Extracting summary/aggregate information from the document
- Example: Extract project details from grant document (one project per document)

IMPORTANT: 'For each X, extract...' means FLAT if X is the document itself,
NESTED if X refers to multiple items within the document.

Requirements:
{user_description}
"""

In [11]:
!pip install fitz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.9/425.9 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.6 MB/s eta 0:00:00


In [1]:
pip install tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.8 MB/s eta 0:00:00


In [2]:
import os
import base64
from io import BytesIO
from typing import List, Optional
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI
from PIL import Image
import fitz  # PyMuPDF for PDF processing

load_dotenv()


class VisionParser:
    """
    A PDF parser that converts PDFs to images and uses OpenAI Vision API
    (Azure or standard) to extract content in markdown format.
    """

    # Default prompt optimized for table preservation
    DEFAULT_TABLE_PROMPT = """
Convert this document page to accurate markdown format. Follow these rules STRICTLY:

**CRITICAL RULES:**
1. **NO HALLUCINATION**: Only output content that is actually visible on the page
2. **NO EMPTY ROWS**: Do NOT create empty table rows. If you see a table, only include rows with actual data
3. **STOP when content ends**: When you reach the end of visible content, STOP. Do not continue with empty rows

**Formatting Requirements:**
- Tables: Use markdown table syntax with | separators
- Multi-row cells: Keep item descriptions/notes in the same row as the item data
- Table continuations: If a table continues from a previous page, continue it without repeating headers
- Images: Interpret images, charts, graphs at at their proper places
- Preserve ALL visible text: headers, data, footers, page numbers, everything
- Keep numbers, dates, and text exactly as shown
- Maintain document structure and layout
- Keep the items at the same location as they are in the original document

**What to include:**
- All table data
- All section headings, text paragraphs
- Interpretation of all images (if any)

Return ONLY the markdown content, no explanations.
"""

    def __init__(
        self,
        openai_config: dict,
        custom_prompt: str = None,
        use_context: bool = True,
        dpi: int = 300,
        clean_output: bool = True
    ):
        """
        Initialize the VisionParser with all configuration options.

        Args:
            openai_config: Dictionary containing OpenAI configuration
                For Azure:
                    - use_azure: True
                    - api_key: Azure OpenAI API key
                    - azure_endpoint: Azure OpenAI endpoint URL
                    - api_version: API version
                    - model: Deployment name
                For Standard OpenAI:
                    - use_azure: False
                    - api_key: OpenAI API key
                    - model: Model name (e.g., 'gpt-4o-2024-11-20')
            custom_prompt: Optional custom instructions for the vision model (uses DEFAULT_TABLE_PROMPT if None)
            use_context: Whether to provide previous page context for multi-page documents (default: True)
            dpi: Image resolution for PDF conversion (default: 300)
            clean_output: Enable LLM post-processing to clean and merge tables (default: True)
        """
        self.config = openai_config
        self.custom_prompt = custom_prompt or self._get_default_prompt()
        self.use_context = use_context
        self.dpi = dpi
        self.clean_output = clean_output
        self.use_azure = openai_config.get('use_azure', True)

        # Initialize appropriate OpenAI client
        if self.use_azure:
            # Extract base endpoint (remove deployment path if present)
            endpoint = openai_config['azure_endpoint']
            if '/openai/deployments/' in endpoint:
                # Extract base endpoint before /openai/deployments/
                endpoint = endpoint.split('/openai/deployments/')[0]

            # Initialize Azure OpenAI client
            self.client = AzureOpenAI(
                api_key=openai_config['api_key'],
                api_version=openai_config['api_version'],
                azure_endpoint=endpoint
            )
        else:
            # Initialize standard OpenAI client
            self.client = OpenAI(
                api_key=openai_config['api_key']
            )

        self.model = openai_config['model']

    def _get_default_prompt(self) -> str:
        """Get default prompt for markdown extraction."""
        return """
Please convert this document page to markdown format with the following requirements:

1. Preserve ALL content exactly as it appears
2. Maintain the document structure and hierarchy
3. For tables:
   - Use proper markdown table syntax with | separators
   - If this page continues a table from the previous page, continue the table seamlessly
   - Do NOT repeat table headers unless they appear on this page
   - Preserve multi-row cells by repeating content or using appropriate formatting
   - Maintain column alignment
   - Keep all headers and data intact
   - For item descriptions or notes within table cells, keep them in the same row
4. Preserve formatting like bold, italic, lists, etc.
5. For images or charts, describe them briefly in [Image: description] format
6. Maintain the reading order and layout flow
7. Keep numbers, dates, and special characters exactly as shown

Return ONLY the markdown content, no explanations.
"""

    def _pdf_to_images(self, pdf_path: str, dpi: int = 300) -> List[Image.Image]:
        """
        Convert PDF pages to images using PyMuPDF (fitz).
        Images are kept in memory as PIL Image objects.

        Args:
            pdf_path: Path to the PDF file
            dpi: Resolution for image conversion (default: 300)

        Returns:
            List of PIL Image objects
        """
        print(f"Converting PDF to images using PyMuPDF (DPI: {dpi})...")
        images = []
        doc = fitz.open(pdf_path)  # Open the PDF

        # Calculate zoom factor from DPI (72 is the default DPI in PDFs)
        zoom = dpi / 72
        mat = fitz.Matrix(zoom, zoom)

        for page_num in range(len(doc)):
            # Render page to pixmap with specified DPI
            pix = doc[page_num].get_pixmap(matrix=mat)
            # Convert to PIL Image
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            images.append(img)

        doc.close()
        print(f"Converted {len(images)} pages to images")
        return images

    def _image_to_base64(self, image: Image.Image) -> str:
        """
        Convert PIL Image to base64 string.

        Args:
            image: PIL Image object

        Returns:
            Base64 encoded string
        """
        buffered = BytesIO()
        image.save(buffered, format="PNG")
        return base64.b64encode(buffered.getvalue()).decode('utf-8')

    ##compress before encoding
    # def _image_to_base64(self, image: Image.Image) -> str:
    #     buffered = BytesIO()
    #     # Reduce quality/size while preserving readability
    #     image = image.resize((int(image.width * 0.8), int(image.height * 0.8)), Image.LANCZOS)
    #     image.save(buffered, format="JPEG", quality=85, optimize=True)
    #     return base64.b64encode(buffered.getvalue()).decode('utf-8')

    def _parse_image_with_vision(self, image: Image.Image, page_num: int, previous_context: str = None) -> str:
        """
        Parse a single image using Azure OpenAI Vision API.

        Args:
            image: PIL Image object
            page_num: Page number (for logging)
            previous_context: Context from previous page(s) to help with continuations

        Returns:
            Markdown content extracted from the image
        """
        print(f"Parsing page {page_num} with vision model...")

        # Convert image to base64
        base64_image = self._image_to_base64(image)

        # Build the prompt with context if available
        prompt_text = self.custom_prompt
        if previous_context and self.use_context:
            prompt_text = f"""
{self.custom_prompt}

CONTEXT FROM PREVIOUS PAGE:
The previous page has the following content:
```
{previous_context}
```

If this page continues a table or section from the previous page, continue it seamlessly without repeating headers.
"""

        # Create the vision API request
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt_text
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/png;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=16000,  # Increased to capture all content
            temperature=0
        )

        markdown_content = response.choices[0].message.content
        return markdown_content

    def _clean_markdown_with_llm(self, markdown_pages: List[str]) -> str:
        """
        Use LLM to clean up and merge markdown from multiple pages.

        Args:
            markdown_pages: List of markdown strings from each page

        Returns:
            Cleaned and merged markdown
        """
        print("Cleaning and merging markdown ...")

        # Combine pages with separators
        combined = "\n\n---PAGE_BREAK---\n\n".join(markdown_pages)

        cleanup_prompt = """
You are a document processing expert. Clean up and merge this multi-page markdown document.

TASKS:
1. **Remove artifacts**: Delete any empty table rows or hallucinated content (rows with only pipe separators and no data)
2. **Merge broken tables**: When a table continues across pages (separated by ---PAGE_BREAK---):
   - Keep only ONE table header
   - Merge all data rows into a single continuous table
   - Remove page break markers within tables
3. **Handle incomplete rows**: If a table row is split across pages, merge it into a complete row
4. **Preserve all real content**: Keep all actual data, headers, footers, and text
5. **Clean up formatting**: Ensure proper markdown syntax throughout
6. **Do NOT hallucinate**: Only output what you see in the input

INPUT MARKDOWN:
```markdown
{markdown}
```

OUTPUT: Return ONLY the cleaned, merged markdown. No explanations, no code blocks wrapper.
"""

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": cleanup_prompt.format(markdown=combined)
                }
            ],
            max_tokens=16000,
            temperature=0
        )

        cleaned_markdown = response.choices[0].message.content

        # Remove any markdown code block wrappers if present
        if cleaned_markdown.startswith("```markdown"):
            cleaned_markdown = cleaned_markdown.replace("```markdown", "", 1)
        if cleaned_markdown.startswith("```"):
            cleaned_markdown = cleaned_markdown.replace("```", "", 1)
        if cleaned_markdown.endswith("```"):
            cleaned_markdown = cleaned_markdown.rsplit("```", 1)[0]

        return cleaned_markdown.strip()

    def convert_pdf(self, pdf_path: str) -> List[str]:
        """
        Convert PDF to markdown using vision API.

        Args:
            pdf_path: Path to the PDF file

        Returns:
            List of markdown strings, one per page (or single cleaned string if clean_output=True)
        """
        # Convert PDF to images using configured DPI
        images = self._pdf_to_images(pdf_path, self.dpi)

        # Parse each image with context from previous page
        markdown_pages = []
        for i, image in enumerate(images, 1):
            # Get context from previous page if available
            previous_context = markdown_pages[-1] if markdown_pages and self.use_context else None

            # Parse with context
            markdown = self._parse_image_with_vision(image, i, previous_context)
            markdown_pages.append(markdown)

        # Post-process with LLM to clean up and merge if requested
        if self.clean_output and len(markdown_pages) > 1:
            cleaned = self._clean_markdown_with_llm(markdown_pages)
            return [cleaned]  # Return as single-item list for consistency

        return markdown_pages

    def save_markdown(self, markdown_pages: List[str], output_path: str,
                     separator: str = "\n\n---\n\n"):
        """
        Save markdown pages to a file.

        Args:
            markdown_pages: List of markdown strings
            output_path: Path to save the markdown file
            separator: Separator between pages (default: horizontal rule, only used if multiple pages)
        """
        # If only one page (e.g., already cleaned), save directly
        if len(markdown_pages) == 1:
            combined_markdown = markdown_pages[0]
        else:
            # Combine all pages with separator
            combined_markdown = separator.join(markdown_pages)

        # Save to file
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(combined_markdown)

        print(f"Markdown saved to: {output_path}")





In [4]:
"""
Shared API configuration for OpenAI and Azure OpenAI.

This module provides reusable configuration utilities for creating
OpenAI/Azure OpenAI clients across different extraction modules.
"""

import os
from dotenv import load_dotenv
from openai import OpenAI, AzureOpenAI

# Load environment variables
load_dotenv()


def get_openai_config(use_azure: bool = True) -> dict:
    """
    Get OpenAI configuration based on whether to use Azure or standard OpenAI.

    Args:
        use_azure: If True, use Azure OpenAI. If False, use standard OpenAI API.

    Returns:
        Configuration dictionary with appropriate settings

    Example:
        >>> config = get_openai_config(use_azure=True)
        >>> # Returns Azure config with deployment name
        >>> config = get_openai_config(use_azure=False)
        >>> # Returns OpenAI config with model name
    """
    if use_azure:
        return {
            'use_azure': True,
            'api_key': os.getenv("AZURE_API_KEY"),
            'azure_endpoint': "https://haagahelia-poc-gaik.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?",
            'azure_audio_endpoint': "https://haagahelia-poc-gaik.openai.azure.com/openai/deployments/whisper/audio/translations?api-version=2024-06-01",
            'api_version': "2024-12-01-preview",
            'model': 'gpt-4.1',   #gpt-5    #gpt-4.1
        }
    else:
        return {
            'use_azure': False,
            'api_key': os.getenv("OPENAI_API_KEY"),
            'model': 'gpt-4.1-2025-04-14',  #gpt-5-2025-08-07    #gpt-4.1-2025-04-14
        }


def create_openai_client(config: dict):
    """
    Create an OpenAI or Azure OpenAI client based on configuration.

    Args:
        config: Configuration dictionary from get_openai_config()

    Returns:
        OpenAI or AzureOpenAI client instance

    Example:
        >>> config = get_openai_config(use_azure=True)
        >>> client = create_openai_client(config)
    """
    if config.get('use_azure', False):
        return AzureOpenAI(
            api_key=config['api_key'],
            api_version=config['api_version'],
            azure_endpoint=config['azure_endpoint'],
        )
    else:
        return OpenAI(api_key=config['api_key'])

In [5]:
config = get_openai_config(use_azure=True)
parser = VisionParser(
    openai_config=config,
    use_context=True,      # Enable inter-page context
    dpi=300,               # Image resolution (200-300 recommended)
    clean_output=True      # Enable LLM-powered table merging
)

# Convert PDF to markdown
markdown_pages = parser.convert_pdf("document.pdf")
parser.save_markdown(markdown_pages, "output/document.md")

In [6]:
DEFAULT_TABLE_PROMPT = """
Convert this document page to markdown format.

CRITICAL RULES:
1. Preserve ALL tables exactly as shown with proper markdown table syntax
2. Do NOT add empty rows that are not visible in the image
3. If a table continues from a previous page, continue it without repeating headers
4. Use | for table columns and separate header row with |---|---|
5. Extract all text content accurately
6. Maintain document structure (headings, lists, paragraphs)

Do not hallucinate content. Only extract what is actually visible.
"""

In [7]:
def _clean_markdown_with_llm(self, markdown_pages: List[str]) -> str:
    # Combine pages with separators
    combined = "\n\n---PAGE_BREAK---\n\n".join(markdown_pages)

    # LLM prompt to clean and merge
    cleaning_prompt = """
You are given a multi-page markdown document with tables that may span pages.

Tasks:
1. Merge tables that are split across PAGE_BREAK markers
2. Remove empty table rows (rows with only | | | structure)
3. Handle incomplete rows at page boundaries
4. Keep all other content unchanged

Return the cleaned markdown.
"""
    # ... LLM processing
    return cleaned_markdown

In [9]:
"""
Dynamic Schema Extraction with Structured Outputs

Schema.py extracts structured data from documents using LLMs,
automatically generates Pydantic schemas from natural language requirements, and
extracts type-safe data with validation.

Main Interface:
    from schema_generator import SchemaGenerator

    generator = SchemaGenerator(use_azure=True)
    results = generator.extract(
        user_requirements="Extract invoice number and total...",
        documents=["document text"]
    )
"""

from __future__ import annotations

import os
import re
import time
from decimal import Decimal, InvalidOperation
from datetime import datetime
from typing import Annotated, Literal, Optional, List, Any, Dict, get_origin, get_args
from openai import OpenAI, AzureOpenAI
from openai import APIError, RateLimitError, APITimeoutError

try:
    # Pydantic v2 style config (preferred)
    from pydantic import ConfigDict
    _HAS_V2 = True
except Exception:
    _HAS_V2 = False


from pydantic import (
    BaseModel,
    Field,
    field_validator,
    create_model,
    ConfigDict,
    constr,
)

# -----------------------------------------------------------------------------
# Setup
# -----------------------------------------------------------------------------

SYSTEM_PARSER = (
    "You convert text into strictly structured data according to the provided schema. "
    "Never invent values. If uncertain or missing, return null. "
    "Do not include explanations or extra keys or extra fields."
)

# -----------------------------------------------------------------------------
# Retry & call helpers
# -----------------------------------------------------------------------------

def _with_retries(call, tries: int = 4):
    for i in range(tries):
        try:
            return call()
        except (RateLimitError, APITimeoutError, APIError) as e:
            if i == tries - 1:
                raise
            time.sleep(2 ** i)  # backoff


def _parse_with(*, client, model: str, messages: list[dict], response_format: type[BaseModel]):
    """
    Wraps client.beta.chat.completions.parse in a retry + deterministic settings.

    Args:
        client: OpenAI or AzureOpenAI client instance
        model: Model name to use
        messages: Messages to send
        response_format: Pydantic model for structured output
    """
    return _with_retries(
        lambda: client.beta.chat.completions.parse(
            model=model,
            messages=messages,
            response_format=response_format,
            temperature=0,
            top_p=1.0,
            seed=12345,
            timeout=30,
        )
    )

# -----------------------------------------------------------------------------
#Fixed schema for parsing the user's extraction requirements
# -----------------------------------------------------------------------------

AllowedTypes = Literal[
    "str", "int", "float", "bool", "list[str]", "date", "decimal", "list[dict]"
]


class FieldSpec(BaseModel):
    """Specification for a single field to extract."""

    field_name: str = Field(description="snake_case field name, must start with a letter")
    field_type: AllowedTypes = Field(description="Type of the field")
    description: str
    required: bool = True
    enum: Optional[list[str]] = Field(default=None, description="Allowed values (if enumerated)")
    pattern: Optional[str] = Field(default=None, description="Regex to validate strings (optional)")
    format: Optional[Literal["iso-date", "currency-eur"]] = Field(default=None)

    @field_validator("field_name")
    @classmethod
    def _snake_case(cls, v: str) -> str:
        v2 = re.sub(r"[^a-zA-Z0-9]+", "_", v).strip("_").lower()
        if not re.match(r"^[a-z][a-z0-9_]*$", v2 or ""):
            raise ValueError("field_name must be snake_case and start with a letter")
        return v2

    @field_validator("enum")
    @classmethod
    def _enum_nonempty(cls, v: Optional[list[str]]) -> Optional[list[str]]:
        if v is not None and len(v) == 0:
            raise ValueError("enum must be a non-empty list when provided")
        return v


class ExtractionRequirements(BaseModel):
    """Parsed extraction requirements from user input."""

    use_case_name: str
    fields: list[FieldSpec]

    @field_validator("fields")
    @classmethod
    def _unique_names(cls, fields: list[FieldSpec]) -> list[FieldSpec]:
        seen = set()
        for f in fields:
            if f.field_name in seen:
                raise ValueError(f"Duplicate field_name: {f.field_name}")
            seen.add(f.field_name)
        return fields

# -----------------------------------------------------------------------------
# Structure Detection for Nested vs Flat Schemas
# -----------------------------------------------------------------------------

class StructureAnalysis(BaseModel):
    """Analysis of whether the extraction requires nested or flat structure."""

    structure_type: Literal["flat", "nested_list"] = Field(
        description="Type of structure: 'flat' for single object, 'nested_list' for array of items"
    )
    parent_container_name: str = Field(
        description="Name for the parent container (e.g., 'items', 'records', 'entries')"
    )
    parent_description: str = Field(
        description="Description of what the parent container holds"
    )
    item_description: str = Field(
        description="Description of extraction requirements for each individual item (if nested)"
    )
    reasoning: str = Field(
        description="Brief explanation of why this structure was chosen"
    )


def detect_structure_type(user_description: str, *, client=None, model: str = None) -> StructureAnalysis:
    """
    Analyze if the extraction requires a nested list structure or flat structure.
    """
    if client is None:
        config = get_openai_config(use_azure=True)
        client = create_openai_client(config)
        model = model if model else config['model']
    elif model is None:
        raise ValueError("model must be provided when client is specified")

    resp = _parse_with(
        client=client,
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PARSER},
            {
                "role": "user",
                "content": (
                    "Analyze the following extraction requirements and determine the output structure.\n\n"
                    "Use NESTED_LIST when:\n"
                    "- The DOCUMENT contains multiple items/records/rows to extract\n"
                    "- Instructions mention 'multiple items IN THE DOCUMENT', 'list of items', 'table of records'\n"
                    "- 'one line per item', 'one row per record', 'repeat for each entry'\n"
                    "- Document is structured as a table, list, or collection of similar items\n"
                    "- Example: Extract all products from an invoice (multiple products in one invoice)\n\n"
                    "Use FLAT when:\n"
                    "- ONE record per document (even if processing multiple documents)\n"
                    "- 'for each document', 'from each document', 'per document'\n"
                    "- Document describes a SINGLE entity (e.g., one project, one invoice, one person)\n"
                    "- Extracting summary/aggregate information from the document\n"
                    "- Example: Extract project details from grant document (one project per document)\n\n"
                    "IMPORTANT: 'For each X, extract...' means FLAT if X is the document itself, NESTED if X refers to multiple items within the document.\n\n"
                    "Requirements:\n```txt\n" + user_description + "\n```"
                ),
            },
        ],
        response_format=StructureAnalysis,
    )
    analysis = resp.choices[0].message.parsed
    if getattr(resp, "usage", None):
        print(f"[detect_structure_type] tokens={resp.usage.total_tokens}")
    return analysis


def parse_nested_requirements(
    user_description: str,
    *,
    client=None,
    model: str = None
) -> tuple[type[BaseModel], ExtractionRequirements, StructureAnalysis]:
    """
    Stage 2: Parse nested requirements by:
    1. Detecting structure type
    2. Parsing item-level fields
    3. Creating nested parent model

    Returns: (ParentModel, item_requirements, structure_analysis)
    """
    if client is None:
        config = get_openai_config(use_azure=True)
        client = create_openai_client(config)
        model = model if model else config['model']
    elif model is None:
        raise ValueError("model must be provided when client is specified")

    print("Analyzing structure type...")
    analysis = detect_structure_type(user_description, client=client, model=model)

    print(f"✓ Structure type: {analysis.structure_type}")
    print(f"  Reasoning: {analysis.reasoning}")

    if analysis.structure_type == "flat":
        # Just parse as flat requirements
        print("  → Using flat structure")
        requirements = parse_user_requirements(user_description, client=client, model=model)
        extraction_model = create_extraction_model(requirements)
        return extraction_model, requirements, analysis

    # Nested structure
    print(f"  → Using nested structure with '{analysis.parent_container_name}' field")
    print(f"  Parent: {analysis.parent_description}")

    print("\nParsing item-level fields...")
    item_requirements = parse_user_requirements(analysis.item_description, client=client, model=model)

    print(f"✓ Identified {len(item_requirements.fields)} fields per item")
    print(f"  Fields: {[f.field_name for f in item_requirements.fields]}")

    print("\nCreating nested Pydantic model...")
    ItemModel = create_extraction_model(item_requirements)

    # Create parent model with items list
    suffix = "_Collection"
    base_name = sanitize_model_name(item_requirements.use_case_name, suffix=suffix)
    model_name = base_name + suffix

    ParentModel = create_model(
        model_name,
        __config__=ConfigDict(extra="forbid"),
        __doc__=f"Collection of {item_requirements.use_case_name} items",
        **{
            analysis.parent_container_name: (
                List[ItemModel],
                Field(description=analysis.parent_description)
            )
        }
    )

    print(f"✓ Created nested model: {ParentModel.__name__}")
    print(f"  Container field: '{analysis.parent_container_name}' (List[{ItemModel.__name__}])")

    return ParentModel, item_requirements, analysis


# -----------------------------------------------------------------------------
# Parse the user's natural language into field specs
# -----------------------------------------------------------------------------

def parse_user_requirements(user_description: str, *, client=None, model: str = None) -> ExtractionRequirements:
    """
    Parse the extraction requirements from the user's natural language using structured outputs.
    """
    if client is None:
        config = get_openai_config(use_azure=True)
        client = create_openai_client(config)
        model = model if model else config['model']
    elif model is None:
        raise ValueError("model must be provided when client is specified")

    resp = _parse_with(
        client=client,
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PARSER},
            {
                "role": "user",
                "content": "Parse the extraction requirements below into the target schema.\n"
                           "If a field cannot be identified reliably, omit it.\n"
                           "```txt\n" + user_description + "\n```"
            },
        ],
        response_format=ExtractionRequirements,
    )
    req = resp.choices[0].message.parsed
    if getattr(resp, "usage", None):
        print(f"[parse_user_requirements] tokens={resp.usage.total_tokens}")
    return req

# -----------------------------------------------------------------------------
# Create dynamic Pydantic model from field specs
# -----------------------------------------------------------------------------

def sanitize_model_name(name: str, suffix: str = "") -> str:
    """
    Sanitize model name following OpenAI requirements.
    Only alphanumeric, underscores, and hyphens are allowed.
    Ensures final name (with suffix) is <= 64 chars.

    Args:
        name: The base name to sanitize
        suffix: Optional suffix to add (e.g., "_Extraction", "_Collection")

    Returns:
        Sanitized name that when combined with suffix is <= 64 chars
    """
    # Replace invalid characters with underscores
    s = re.sub(r"[^a-zA-Z0-9_-]", "_", name)
    # Remove consecutive underscores
    s = re.sub(r"_+", "_", s).strip("_")

    # Remove leading/trailing underscores
    s = s.strip("_")

    # Remove suffix from name if it already exists (avoid duplication)
    if suffix and s.endswith(suffix.lstrip("_")):
        s = s[:-len(suffix.lstrip("_"))].rstrip("_")

    # Ensure final name fits within 64 char limit
    max_length = 64 - len(suffix)
    if len(s) > max_length:
        s = s[:max_length].rstrip("_")

    return s if s else "Dynamic"

def create_extraction_model(requirements: ExtractionRequirements) -> type[BaseModel]:
    """
    Create a Pydantic model dynamically from field specifications (strict).
    - Forbid extra/unknown keys.
    - Apply enums, regex patterns, and formats where applicable.
    """
    # Map string type names to actual Python types
    base_types = {
        "str": str,
        "int": int,
        "float": float,
        "bool": bool,
        "list[str]": list[str],
        "date": str,
        "decimal": Decimal,
        "list[dict]": list[dict],  # for nested structures like items
    }

    field_defs: dict[str, tuple[object, Field]] = {}

    for f in requirements.fields:
        py_type = base_types[f.field_type]

        # Constrain strings when possible
        annotated: object = py_type
        if f.field_type == "str" and f.pattern:
            annotated = Annotated[str, constr(pattern=f.pattern)]
        elif f.field_type == "decimal":
            annotated = Decimal  # leave numeric constraints to normalization/validation

        # Enums → Literal[...] for strict checking
        if f.enum:
            # Build a Literal[...] dynamically; acceptable for runtime checks
            annotated = Literal[tuple(f.enum)]  # type: ignore[misc,call-arg]

        # Optionality
        required_default = ... if f.required else None
        typ = annotated if f.required else (annotated | None)

        field_defs[f.field_name] = (
            typ,
            Field(default=required_default, description=f.description),
        )

    suffix = "_Extraction"
    base_name = sanitize_model_name(requirements.use_case_name, suffix=suffix)
    model_name = base_name + suffix

    DynamicModel = create_model(
        model_name,
        __config__=ConfigDict(extra="forbid"),
        __doc__=f"Extraction model for {requirements.use_case_name}",
        **field_defs,
    )
    return DynamicModel

# -----------------------------------------------------------------------------
# Normalization helpers (post-LLM)
# -----------------------------------------------------------------------------

def _to_iso_date(s: str) -> str:
    s = s.strip()
    for fmt in ("%Y-%m-%d", "%d.%m.%Y", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(s, fmt).date().isoformat()
        except ValueError:
            continue
    return s  # leave as-is if unparseable


def _normalize_record(data: dict, req: ExtractionRequirements) -> dict:
    spec_by_name = {f.field_name: f for f in req.fields}
    out = {}
    for k, v in data.items():
        spec = spec_by_name.get(k)
        if spec is None or v is None:
            out[k] = v
            continue

        if spec.field_type == "date" and isinstance(v, str):
            out[k] = _to_iso_date(v)
        elif spec.field_type == "list[str]":
            if isinstance(v, str):
                out[k] = [s.strip() for s in re.split(r"[;,]", v) if s.strip()]
            elif isinstance(v, list):
                out[k] = [str(x).strip() for x in v]
            else:
                out[k] = v
        else:
            out[k] = v
    return out


# -----------------------------------------------------------------------------
# Helper: Pretty print Pydantic model schema
# -----------------------------------------------------------------------------

def print_pydantic_schema(model: type[BaseModel], title: str = "Generated Pydantic Schema") -> None:
    """
    Print the exact Pydantic model as Python class definition.
    For nested structures, prints both the inner model and outer container model.
    """
    from typing import get_origin, get_args
    import inspect

    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}\n")

    # Collect all models to print (handle nested structures)
    models_to_print = []

    # Check if this model has nested Pydantic models
    for field_name, field_info in model.model_fields.items():
        annotation = field_info.annotation

        # Check for List[SomeModel] pattern
        origin = get_origin(annotation)
        if origin is list:
            args = get_args(annotation)
            if args and len(args) > 0:
                inner_type = args[0]
                # Check if it's a Pydantic model
                if inspect.isclass(inner_type) and issubclass(inner_type, BaseModel):
                    models_to_print.append(inner_type)

    # Print inner models first
    for inner_model in models_to_print:
        _print_single_model(inner_model)
        print()

    # Print the main model
    _print_single_model(model)

    print(f"\n{'='*80}\n")


def _print_single_model(model: type[BaseModel]) -> None:
    """Helper to print a single Pydantic model."""
    # Class definition
    print(f"class {model.__name__}(BaseModel):")

    # Docstring
    if model.__doc__:
        print(f'    """{model.__doc__}"""')

    # Config
    if hasattr(model, 'model_config'):
        config = model.model_config
        if config.get('extra') == 'forbid':
            print(f"    model_config = ConfigDict(extra='forbid')")

    print()

    # Fields
    for field_name, field_info in model.model_fields.items():
        # Get type annotation
        annotation = field_info.annotation
        type_str = str(annotation).replace("typing.", "").replace("<class '", "").replace("'>", "")

        # Clean up the type string for better readability
        type_str = type_str.replace("schema_generator.", "")

        # Check if required
        is_required = field_info.is_required()

        # Get description
        description = field_info.description

        # Build field definition
        if is_required:
            if description:
                print(f'    {field_name}: {type_str} = Field(description="{description}")')
            else:
                print(f'    {field_name}: {type_str}')
        else:
            if description:
                print(f'    {field_name}: {type_str} = Field(None, description="{description}")')
            else:
                print(f'    {field_name}: {type_str} = None')


# -----------------------------------------------------------------------------
# REUSABLE CLASS INTERFACE
# -----------------------------------------------------------------------------

class SchemaGenerator:
    """
    Generates Pydantic schemas from natural language requirements.

    Automatically detects nested vs flat data structures and generates
    appropriate Pydantic models for structured data extraction.

    Features:
    - Smart structure detection (flat vs nested)
    - Type-safe Pydantic model generation
    - Support for Azure OpenAI and OpenAI
    - Field specification parsing (types, enums, patterns)

    Usage:
        # Create config once
        config = get_openai_config(use_azure=True)  # or use_azure=False for standard OpenAI

        # Initialize with config
        generator = SchemaGenerator(config=config)

        # Generate schema from requirements
        schema = generator.generate_schema(
            user_requirements="Extract invoice number, amount, and date..."
        )

        # Access generated models and requirements
        print(generator.extraction_model)
        print(generator.item_requirements)
        print(generator.get_schema_info())
    """

    def __init__(self, config: dict, model: Optional[str] = None):
        """
        Initialize the SchemaGenerator.

        Args:
            config: OpenAI configuration dict from get_openai_config()
            model: Optional model name override
        """
        self.config = config
        self.model = model if model else self.config['model']
        self.client = create_openai_client(self.config)
        self.extraction_model = None
        self.item_requirements = None
        self.structure_analysis = None

    def analyze_structure(self, user_requirements: str) -> StructureAnalysis:
        """
        Analyze if the requirements need nested or flat structure.

        Args:
            user_requirements: Natural language description of extraction task

        Returns:
            StructureAnalysis with structure type and descriptions
        """
        self.structure_analysis = detect_structure_type(user_requirements, client=self.client, model=self.model)
        return self.structure_analysis

    def generate_schema(self, user_requirements: str) -> type[BaseModel]:
        """
        Generate Pydantic schema from natural language requirements.

        Args:
            user_requirements: Natural language description of fields to extract

        Returns:
            Generated Pydantic model class (nested or flat)
        """
        print("Generating schema from requirements...")
        self.extraction_model, self.item_requirements, self.structure_analysis = parse_nested_requirements(
            user_requirements,
            client=self.client,
            model=self.model
        )

        # Print the generated Pydantic model
        print("\n" + "="*80)
        print("GENERATED PYDANTIC MODEL")
        print("="*80)
        print_pydantic_schema(self.extraction_model, title="Extraction Schema")

        return self.extraction_model

    def get_schema_info(self) -> dict:
        """
        Get information about the generated schema.

        Returns:
            Dict with schema information
        """
        if not self.extraction_model:
            return {"error": "No schema generated yet. Call generate_schema() first."}

        return {
            "model_name": self.extraction_model.__name__,
            "structure_type": self.structure_analysis.structure_type if self.structure_analysis else "unknown",
            "fields": [f.field_name for f in self.item_requirements.fields] if self.item_requirements else [],
            "field_count": len(self.item_requirements.fields) if self.item_requirements else 0
        }



In [11]:
config = get_openai_config(use_azure=True)
generator = SchemaGenerator(config=config)

requirements = """
Extract invoice line items:
- Item number (string, required)
- Description (string, required)
- Quantity (integer, required)
- Unit price (decimal, required)
- Total amount (decimal, required)
"""

# Generate the complete schema
schema = generator.generate_schema(user_requirements=requirements)

# Access the generated components
print(generator.extraction_model)      # The Pydantic model class
print(generator.item_requirements)     # Field specifications
print(generator.structure_analysis)    # Structure detection result

In [12]:
def create_extraction_model(requirements: ExtractionRequirements) -> type[BaseModel]:
    """
    Dynamically create a Pydantic model from field specifications.
    """
    fields = {}

    for spec in requirements.fields:
        # Map field types
        if spec.field_type == "str":
            if spec.enum:
                # Create Literal type for enums
                field_type = Literal[tuple(spec.enum)]
            elif spec.pattern:
                # Add regex pattern constraint
                field_type = Annotated[str, constr(pattern=spec.pattern)]
            else:
                field_type = str
        elif spec.field_type == "int":
            field_type = int
        elif spec.field_type == "decimal":
            field_type = Decimal
        # ... more type mappings

        # Create Field with description
        default = ... if spec.required else None
        fields[spec.field_name] = (
            field_type,
            Field(default=default, description=spec.description)
        )

    # Create model with strict validation
    model = create_model(
        sanitize_model_name(requirements.use_case_name),
        __config__=ConfigDict(extra='forbid'),
        **fields
    )

    return model

In [13]:
class InvoiceLineItem_Extraction(BaseModel):
    model_config = ConfigDict(extra='forbid')
    item_number: str = Field(description="Item number")
    description: str = Field(description="Description")
    quantity: int = Field(description="Quantity")
    unit_price: Decimal = Field(description="Unit price")
    total_amount: Decimal = Field(description="Total amount")

class InvoiceLineItems_Collection(BaseModel):
    model_config = ConfigDict(extra='forbid')

    items: List[InvoiceLineItem_Extraction] = Field(
        description="List of invoice line items"
    )

In [14]:
config = get_openai_config(use_azure=True)
extractor = DataExtractor(config=config)

# Assume 'schema' and 'requirements' from SchemaGenerator
documents = [
    """
    Invoice #INV-2024-001
    Date: 2024-01-15

    Items:
    1. Widget A - Qty: 5 - Price: $19.99 - Total: $99.95
    2. Gadget B - Qty: 3 - Price: $29.99 - Total: $89.97

    Total: $189.92
    """
]

results = extractor.extract(
    extraction_model=schema,                    # Generated Pydantic model
    requirements=generator.item_requirements,   # Field specifications
    user_requirements=requirements,             # Original text requirements
    documents=documents,
    save_json=True,
    json_path="invoice_extraction.json"
)

print(results)
# [{'items': [{'item_number': '1', 'description': 'Widget A', ...}, ...]}]

In [15]:
def _parse_with(client, model, messages, response_format):
    """
    Call OpenAI with structured output format using Pydantic schema.
    """
    response = client.beta.chat.completions.parse(
        model=model,
        messages=messages,
        response_format=response_format,  # Pydantic model class
        temperature=0,     # Deterministic outputs
        top_p=1.0,
        seed=12345,        # Fixed seed for reproducibility
        timeout=30
    )
    return response

In [16]:
SYSTEM_PARSER = """
You are a data extraction assistant. Your task is to extract structured information
from documents according to the provided schema.

Instructions:
1. Extract data exactly as specified in the requirements
2. Preserve original values when possible
3. For dates, extract in the format present in the document
4. For lists, extract all items mentioned
5. If a field is not present, leave it as null (if optional)
6. Do not invent or hallucinate data
7. Be precise with numbers and decimals

The schema will enforce the output format. Extract accurately.
"""